# ☁️ Maquina-v7 — Cloud PC Linux + LXQt + Apache Guacamole en Google Colab

> Escritorio remoto **GNU/Linux (LXQt)** accesible desde el **navegador** mediante **Apache Guacamole**.
> Sin necesidad de instalar clientes RDP/VNC. Funciona en cualquier navegador web.

**Arquitectura:**
```
Google Colab → Linux → Xvfb → LXQt → x11vnc → guacd → Guacamole → Browser
```

**Pasos:** ejecuta las celdas en orden: 1) Configuracion, 2) Instalar, 3) Iniciar, 4) Estado.

In [ ]:
#@title ⚙️ Configuracion
GITHUB_USER = "jephersonRD"   #@param {type:"string"}
REPO        = "Maquina-v7"    #@param {type:"string"}

USERNAME = "user"             #@param {type:"string"}
PASSWORD = "password"         #@param {type:"string"}
RESOLUTION = "1920x1080"      #@param {type:"string"}
GUAC_PORT = 8080              #@param {type:"integer"}

print("✅ Configuracion lista.")
print(f"   Resolucion: {RESOLUTION}")
print(f"   Puerto Guacamole: {GUAC_PORT}")
print(f"   Usuario: {USERNAME}")

In [ ]:
#@title 📦 Instalar LXQt + Guacamole + Audio
import os

if not os.path.isdir('/content/Maquina-v7'):
    print("📥 Descargando repositorio...")
    !curl -fsSL "https://codeload.github.com/{GITHUB_USER}/{REPO}/tar.gz/refs/heads/main" -o /tmp/maquina-v7.tar.gz
    !tar xzf /tmp/maquina-v7.tar.gz -C /content
    !mv /content/{REPO}-main /content/Maquina-v7
    !rm -f /tmp/maquina-v7.tar.gz

%cd /content/Maquina-v7/scripts

print("\n" + "="*50)
print("📦 [1/4] Instalando LXQt...")
print("="*50)
!bash setup_lxqt.sh {RESOLUTION}

print("\n" + "="*50)
print("🔊 [2/4] Configurando audio...")
print("="*50)
!bash setup_audio.sh

print("\n" + "="*50)
print("🔗 [3/4] Instalando VNC...")
print("="*50)
!bash setup_vnc.sh

print("\n" + "="*50)
print("🌐 [4/4] Instalando Apache Guacamole (puede tardar 5-10 min)...")
print("="*50)
!bash setup_guacamole.sh

print("\n✅ Instalacion completada")

In [ ]:
#@title 🚀 Iniciar Escritorio
%cd /content/Maquina-v7/scripts
!bash start_desktop.sh {RESOLUTION} {USERNAME} {PASSWORD} {GUAC_PORT}

# Detectar URL publica de cloudflared
import re
public_url = ""
try:
    with open("/tmp/cloudflared.log") as f:
        for line in f:
            m = re.search(r'https://[a-zA-Z0-9._-]+\.trycloudflare\.com', line)
            if m:
                public_url = m.group(0)
                break
except FileNotFoundError:
    pass

print("\n" + "="*50)
print("  🌐 COMO ACCEDER:")
print("="*50)
if public_url:
    print(f"  ✅ URL PUBLICA (desde cualquier dispositivo):")
    print(f"     {public_url}")
    print(f"")
print(f"  URL local (solo Colab): http://localhost:{GUAC_PORT}")
print(f"")
print(f"  Usuario : {USERNAME}")
print(f"  Password: {PASSWORD}")
print("="*50)

In [ ]:
#@title 📊 Estado del Sistema
%cd /content/Maquina-v7/scripts
!bash diagnostics.sh

In [ ]:
#@title 🔒 Keep-Alive (anti-apagado de Colab)
import threading, time

def _heartbeat():
    while True:
        time.sleep(60)
        print("♥ keep-alive", time.strftime('%H:%M:%S'))
threading.Thread(target=_heartbeat, daemon=True).start()
print('✅ Keep-alive activado.')
print('⚠️ NO cierres ni ocultes esta pestaña de Colab.')

In [ ]:
#@title 🎮 Instalar Steam (opcional)
INSTALL_STEAM = False  #@param {type:"boolean"}
if INSTALL_STEAM:
    %cd /content/Maquina-v7/scripts
    !bash install_steam.sh
else:
    print('Omitido (INSTALL_STEAM = False).')